# Phase 2 — Long-Document Retrieval

**Weeks 4–6 · Goal:** Retrieve the right pages from 200+ page PDFs using better chunking, hybrid search, and reranking.

## What you will learn

- **Section paths** — breadcrumb context prepended before embedding (`2 > Financial Results > Revenue`)
- **Recursive vs semantic chunking** — split long text without losing coherence
- **Hybrid retrieval** — BM25 sparse + dense vectors fused with RRF (text collection only)
- **Parent expand** — retrieve small child chunks, return full parent context
- **FlashRank** — fast cross-encoder reranking alternative to Phase 1 reranker

> Phase 1 multimodal paths (figures, tables, ColPali pages) are unchanged; Phase 2 enhances **text** retrieval.


In [1]:
import sys
from pathlib import Path

# Run from notebooks/ — project root is one level up
ROOT = Path.cwd()
if not (ROOT / "shared").exists() and (ROOT.parent / "shared").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT.resolve()}")


Project root: C:\Users\dyh\Desktop\RAG


## Feature flags

All features are toggled via `PipelineConfig` or CLI flags on `demo.py`:

| Feature | Config / flag | When |
|---------|---------------|------|
| Recursive chunker | `use_recursive_chunker` / `--recursive-chunk` | Ingest |
| Semantic chunker | `use_semantic_chunker` / `--semantic-chunk` | Ingest |
| Section paths | `use_section_paths` (default on) | Ingest |
| Context enrichment | `use_context_enrichment` (default on) | Ingest |
| Hybrid BM25+dense | `use_hybrid` / `--hybrid` | Query |
| Parent expand | `use_parent_expand` (default on) | Query |
| FlashRank | `use_flashrank` / `--flashrank` | Query |


In [2]:
from phases.phase_02_long_doc_retrieval.chunking.recursive_chunker import RecursiveChunker
from shared.models import ChunkType, DocumentChunk, DocumentType

# Synthetic long section — recursive chunker splits on paragraph/sentence boundaries
long_text = "Configuration management baseline.\n\n" + ("A baseline is a specification approved for use. " * 40)
parent = DocumentChunk(
    id="parent-1",
    doc_id="demo",
    source_path="demo.pdf",
    doc_type=DocumentType.PDF,
    chunk_type=ChunkType.TEXT,
    content=long_text,
    page_number=42,
    section_path="3 > Configuration Management > Baselines",
)
chunker = RecursiveChunker(chunk_size=200, chunk_overlap=30)
children = chunker.chunk([parent])
print(f"Split 1 parent chunk into {len(children)} children")
print(f"First child section_path: {children[0].section_path}")
print(f"First child preview: {children[0].content[:100]}...")


C:\Users\dyh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Split 1 parent chunk into 11 children
First child section_path: 3 > Configuration Management > Baselines
First child preview: Configuration management baseline....


In [3]:
from phases.phase_02_long_doc_retrieval.retrieval.hybrid_retriever import reciprocal_rank_fusion
from shared.models import ChunkType, DocumentChunk, DocumentType, RetrievalStrategy, RetrievedContext

def _ctx(chunk_id: str, content: str, rank: int) -> RetrievedContext:
    chunk = DocumentChunk(
        id=chunk_id,
        doc_id="demo",
        source_path="demo.pdf",
        doc_type=DocumentType.PDF,
        chunk_type=ChunkType.TEXT,
        content=content,
    )
    return RetrievedContext(chunk=chunk, score=1.0 / rank, strategy=RetrievalStrategy.HYBRID, rank=rank)

# RRF merges ranked lists from BM25 and dense search
sparse = [_ctx("c3", "margin table", 1), _ctx("c1", "revenue", 2), _ctx("c4", "footer", 3)]
dense = [_ctx("c1", "revenue", 1), _ctx("c2", "summary", 2), _ctx("c3", "margin table", 3)]
fused = reciprocal_rank_fusion([sparse, dense], k=60)
print("Fused ranking (higher score = better):")
for ctx in fused[:4]:
    print(f"  {ctx.chunk.id}: {ctx.score:.4f}")


Fused ranking (higher score = better):
  c1: 0.0325
  c3: 0.0323
  c2: 0.0161
  c4: 0.0159


## Try on a long document

Use `data/raw/long_report.pdf` (812-page textbook) with hybrid + recursive chunk:

```bash
python phases/phase_01_multimodal_ingestion/demo.py \
  --doc data/raw/long_report.pdf \
  --query "What is a baseline in configuration management?" \
  --hybrid --recursive-chunk --retrieve-only
```


In [4]:
from shared.models import QueryRequest
from shared.pipeline import PipelineConfig, RAGPipeline

# Short demo on sample_report.pdf (long_report.pdf works the same with --hybrid --recursive-chunk)
sample = ROOT / "data/raw/sample_report.pdf"
if not sample.exists():
    print("Run: python scripts/generate_sample_report.py")
else:
    try:
        cfg = PipelineConfig(
            text_only=True,
            use_hybrid=True,
            use_recursive_chunker=True,
            use_section_paths=True,
            use_parent_expand=True,
        )
        pipe = RAGPipeline(config=cfg)
        ing = pipe.ingest(str(sample))
        resp = pipe.query(
            QueryRequest(
                query="What was Q4 2024 revenue and operating margin?",
                doc_id=ing.doc_id,
                top_k=5,
                retrieve_only=True,
            )
        )
        print(f"Hybrid retrieval returned {len(resp.contexts)} contexts")
        for ctx in resp.contexts[:3]:
            print(f"  p{ctx.page_number} [{ctx.chunk_type}] {ctx.content[:80]}...")
    except Exception as exc:
        print(f"Requires Qdrant: {exc}")


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 17988.47it/s]


Requires Qdrant: Install rdflib: pip install -e '.[phase4]'


**Next:** [03_scalable_ingestion.ipynb](03_scalable_ingestion.ipynb) — bulk ingest at scale with Celery and Redis caching.
